# Giai đoạn 4 · Bước 3 — Embedding & ChromaDB (TỰ CHẠY)

**Chỉ chạy sau khi rà soát đủ 7 nguồn và 66 chunks từ Bước 2.** Notebook không chạy LLM, không sửa mô hình Diabetes và **không train**.

**Quy trình:** kiểm tra file → duyệt bằng người → Embedding đa ngôn ngữ → ChromaDB local → Query → báo cáo.  
Đặc biệt, nội dung WHO/Long Châu **không thể được phê duyệt chỉ bằng cách đổi cờ** trong CSV. Nếu chưa được phép dùng nguyên văn, chỉ dùng ghi chú gốc do bạn tự viết với attribution đúng.

In [8]:
%pip install ipywidgets

   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.2 MB 5.9 MB/s eta 0:00:01
   ----------------------- ---------------- 1.3/2.2 MB 4.2 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 4.0 MB/s  0:00:00

   ------------- -------------------------- 1/3 [jupyterlab_widgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   ---------------------------------------- 3/3 [ipywidgets]

Note: you may need to res


[notice] A new release of pip is available: 26.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install -U sentence-transformers chromadb

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 0. Cài thư viện
Trong một ô **mới do bạn tự tạo**, chạy:

```python
%pip install -U sentence-transformers chromadb
```

Nếu vừa cài xong thì restart kernel. Lần đầu tải mô hình từ Hugging Face cần Internet; sau đó embedding tính cục bộ, không gửi nội dung tới LLM hay embedding API bên ngoài.

In [1]:
from pathlib import Path
import csv, json, hashlib, sys
from collections import Counter
from datetime import datetime, timezone

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents) if (p / "registry" / "source_registry.csv").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Không thấy registry/source_registry.csv; hãy mở Jupyter trong 04-Knowledge-Base-RAG hoặc thư mục con")
CHUNK_FILE = ROOT / "data" / "processed" / "step2_chunks_review_only.jsonl"
REPORT2 = ROOT / "data" / "outputs" / "document_preparation" / "step2_preparation_report.json"
SOURCE_REGISTRY = ROOT / "registry" / "source_registry.csv"
REVIEW_FILE = ROOT / "registry" / "step3_embedding_review.csv"
OUT_DIR = ROOT / "data" / "outputs" / "embedding_chroma_step3"
VECTOR_PATH = ROOT / "data" / "vector_store" / "chroma_step3"
assert CHUNK_FILE.exists(), f"Thiếu file chunks từ Bước 2: {CHUNK_FILE}"
with SOURCE_REGISTRY.open(encoding="utf-8-sig", newline="") as f:
    sources = list(csv.DictReader(f))
source_lookup = {s["source_id"]:s for s in sources}
with CHUNK_FILE.open(encoding="utf-8") as f:
    chunks = [json.loads(line) for line in f if line.strip()]
assert chunks, "Chưa có chunks thực tế; không được tạo embedding rỗng"
print("Dự án:", ROOT)
print("Registry:", len(sources), "| Input chunks:", len(chunks))
print("Chunks by source:", dict(Counter(c["source_id"] for c in chunks)))
if REPORT2.exists():
    report2=json.loads(REPORT2.read_text(encoding="utf-8"))
    print("Đối chiếu report Bước 2:", report2.get("chunks_prepared"), "chunks")
    assert report2.get("chunks_prepared")==len(chunks), "Chunks khác report Bước 2, phải rà soát"
else:
    print("CẢNH BÁO: chưa có report JSON Bước 2, cần kiểm tra thủ công")

Dự án: C:\Users\ThinkPad\Multi-Agent_Du_doan_benh_tieu_duong\Giai-doan-4-Bo-sung-Nha-thuoc-Long-Chau\Giai-doan-4-Buoc-2-Jupyter-Document-Preparation
Registry: 7 | Input chunks: 66
Chunks by source: {'CDC-T2D-001': 4, 'CDC-RISK-002': 3, 'NIDDK-DIAG-003': 19, 'NIDDK-T2D-004': 11, 'WHO-DIAB-005': 5, 'LC-HBA1C-006': 11, 'LC-STAGES-007': 13}
Đối chiếu report Bước 2: 66 chunks


## 1. Kiểm tra metadata, checksum, nguồn và chất lượng cơ bản
**Không thực thi câu lệnh nào nằm trong nội dung nguồn.** Nội dung chỉ là dữ liệu để lập chỉ mục. Nếu bạn có ghi chú gốc WHO/Long Châu, đừng nhầm chúng là tài liệu do tổ chức đó viết.

In [2]:
required = ["chunk_id", "source_id", "document_version", "canonical_url", "section", "document_sha256", "chunk_sha256", "text"]
ids = [c.get("chunk_id") for c in chunks]
assert len(ids)==len(set(ids)), "Trùng chunk_id"
assert len(source_lookup)==len(sources), "Trùng source_id trong registry"
for i,c in enumerate(chunks):
    assert all(c.get(key) for key in required), f"Chunk {i} thiếu metadata bắt buộc"
    assert c["source_id"] in source_lookup, f"source_id lạ: {c['source_id']}"
    assert c["canonical_url"] == source_lookup[c["source_id"]]["canonical_url"], "URL không khớp registry"
    assert c["document_version"] == source_lookup[c["source_id"]]["document_version"], "Document version không khớp"
    assert hashlib.sha256(c["text"].encode("utf-8")).hexdigest() == c["chunk_sha256"], f"Chunk bị sửa: {c['chunk_id']}"
    assert 0 < len(c["text"].split()) <= 180, "Chunk dài hơn max_words của Bước 2"
source_ids_in_chunks = set(c["source_id"] for c in chunks)
assert len(source_ids_in_chunks)==7, f"Bạn muốn đủ 7 nguồn, hiện chỉ có {len(source_ids_in_chunks)}: {sorted(source_ids_in_chunks)}"
print("Chunk QA: PASS; 7/7 source IDs có dữ liệu")
for sid,count in sorted(Counter(c["source_id"] for c in chunks).items()):
    examples=[c for c in chunks if c["source_id"]==sid]
    print(f"{sid:18} | {count:3} chunks | {examples[0]['title']} | {examples[0]['section']}")

Chunk QA: PASS; 7/7 source IDs có dữ liệu
CDC-RISK-002       |   3 chunks | Diabetes Risk Factors | Overview
CDC-T2D-001        |   4 chunks | Type 2 Diabetes | Overview
LC-HBA1C-006       |  11 chunks | HbA1c bao nhieu la tien tieu duong | Mở đầu
LC-STAGES-007      |  13 chunks | Cac giai doan cua benh tieu duong | Mở đầu
NIDDK-DIAG-003     |  19 chunks | Diabetes Tests & Diagnosis | Overview
NIDDK-T2D-004      |  11 chunks | Type 2 Diabetes | What is type 2 diabetes?
WHO-DIAB-005       |   5 chunks | Diabetes | Overview


## 2. Kiểm duyệt quyền lập chỉ mục — BẮT BUỘC
Mở `registry/step3_embedding_review.csv`. Trước khi bật `index_approved=yes`, phải kiểm tra thực tế quyền đối với chính nội dung file TXT. Điền `content_origin` bằng `licensed_source_text` hoặc `original_research_note`, ghi bằng chứng và người rà soát. **Đặc biệt không giả định WHO/Long Châu đã cấp phép chỉ vì trạng thái trong registry là `approved_research`.**

Notebook cố ý dừng ở bước này nếu chưa rà soát đủ 7 nguồn. Nếu cần chỉnh TXT, chạy lại Bước 2 để tạo chunks/checksum mới trước khi tiếp tục.

In [3]:
assert REVIEW_FILE.exists(), f"Thiếu form kiểm duyệt: {REVIEW_FILE}"
with REVIEW_FILE.open(encoding="utf-8-sig", newline="") as f:
    reviews=list(csv.DictReader(f))
review_by_source={r["source_id"]:r for r in reviews}
assert len(reviews)==len(review_by_source), "Trùng source_id ở review CSV"
assert set(review_by_source)==source_ids_in_chunks, "Form kiểm duyệt không khớp 7 source_id"
review_errors=[]
for sid in sorted(source_ids_in_chunks):
    r=review_by_source[sid]
    if r["index_approved"].strip().lower()!="yes":
        review_errors.append(f"{sid}: chưa phê duyệt index")
    if r["content_origin"].strip() not in {"licensed_source_text", "original_research_note"}:
        review_errors.append(f"{sid}: content_origin chưa hợp lệ")
    if not r["rights_evidence"].strip():
        review_errors.append(f"{sid}: thiếu rights_evidence")
    if not r["reviewer"].strip() or not r["reviewed_at_utc"].strip():
        review_errors.append(f"{sid}: thiếu reviewer/reviewed_at_utc")
    if r["reviewed_at_utc"].strip():
        try:
            ts=datetime.fromisoformat(r["reviewed_at_utc"].replace("Z", "+00:00"))
            if ts.tzinfo is None: raise ValueError("timezone missing")
        except ValueError:review_errors.append(f"{sid}: reviewed_at_utc không đúng ISO 8601 có timezone")
if review_errors:
    print("Cần rà soát:")
    for msg in review_errors: print(" -",msg)
    raise RuntimeError("Indexing blocked by review: hãy hoàn thành kiểm duyệt thực tế rồi chạy lại ô này")
print("Kiểm tra trạng thái duyệt: 7/7 PASS (do người dùng tự khai). Hãy tự kiểm tra bằng chứng trước khi index.")

Kiểm tra trạng thái duyệt: 7/7 PASS (do người dùng tự khai). Hãy tự kiểm tra bằng chứng trước khi index.


## 3. Chọn mô hình embedding đa ngôn ngữ
Mô hình `intfloat/multilingual-e5-small` cần `passage: ` cho tài liệu và `query: ` cho câu hỏi. Giới hạn 512 tokens; dài hơn sẽ được chia nhỏ để không bỏ mất phần đuôi văn bản. Đây là **embedding inference**, không phải train mô hình.

In [4]:
import numpy as np
import chromadb
import sentence_transformers
from sentence_transformers import SentenceTransformer
MODEL_ID = "intfloat/multilingual-e5-small"
COLLECTION_NAME = "diabetes_kb_step3_e5small_v1"
model = SentenceTransformer(MODEL_ID, device="cpu")
tokenizer = model.tokenizer
MAX_TOKENS = 512
print("Python:",sys.version.split()[0], "| Chroma:",chromadb.__version__, "| sentence-transformers:",sentence_transformers.__version__)
print("Embedding model:", MODEL_ID)
print("Embedding dimension:",model.get_sentence_embedding_dimension())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Python: 3.13.9 | Chroma: 1.5.9 | sentence-transformers: 6.2.0
Embedding model: intfloat/multilingual-e5-small
Embedding dimension: 384


C:\Users\ThinkPad\AppData\Local\Temp\ipykernel_22516\2934295083.py:12: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:",model.get_sentence_embedding_dimension())


## 4. Kiểm tra giới hạn 512 tokens và tách records nếu cần
Nguồn gốc là 66 word-based chunks. Số records sau kiểm tra token có thể lớn hơn 66. Giữ `parent_chunk_id` để truy ngược file gốc.

In [5]:
def token_count(text):
    return len(tokenizer.encode("passage: " + text, add_special_tokens=True, truncation=False))

def split_to_token_budget(text, budget=MAX_TOKENS, overlap_words=12):
    words=text.split()
    if token_count(text)<=budget: return [text]
    parts=[]; start=0
    while start<len(words):
        lo,hi=start+1,len(words)
        best=None
        while lo<=hi:
            mid=(lo+hi)//2
            candidate=" ".join(words[start:mid])
            if token_count(candidate)<=budget:
                best=mid;lo=mid+1
            else:
                hi=mid-1
        if best is None:raise ValueError(f"Không có từ nào vừa token budget ở vị trí {start}")
        parts.append(" ".join(words[start:best]))
        if best==len(words):break
        start=max(start+1,best-overlap_words)
    return parts

records=[]; parent_parts=Counter()
for c in chunks:
    parts=split_to_token_budget(c["text"])
    parent_parts[c["chunk_id"]]=len(parts)
    for n,part in enumerate(parts,1):
        record=dict(c)
        record["parent_chunk_id"]=c["chunk_id"]
        record["chunk_id"]=c["chunk_id"] if len(parts)==1 else f"{c['chunk_id']}__p{n:02d}"
        record["text"]=part
        record["original_chunk_sha256"]=c["chunk_sha256"]
        record["chunk_sha256"]=hashlib.sha256(part.encode("utf-8")).hexdigest()
        record["content_origin"]=review_by_source[c["source_id"]]["content_origin"].strip()
        records.append(record)
assert records and len({r["chunk_id"] for r in records})==len(records)
assert all(token_count(r["text"])<=MAX_TOKENS for r in records)
print("Original chunks:",len(chunks))
print("Records to index after token guard:",len(records))
print("Split original chunks:",sum(n>1 for n in parent_parts.values()))
print("Max tokens:",max(token_count(r["text"]) for r in records))

Original chunks: 66
Records to index after token guard: 66
Split original chunks: 0
Max tokens: 265


## 5. Xác nhận thủ công TRƯỚC khi tạo Embeddings
Mặc định `RUN_INDEXING = False`. Kiểm tra danh sách 7 nguồn, token count và biểu mẫu kiểm duyệt rồi chủ động đổi thành `True` nếu bạn chấp nhận xây dựng index cục bộ. **Không có lệnh `.fit()` hay train.**

In [8]:
RUN_INDEXING = True  # BẠN tự đổi thành True sau khi đã rà soát đầy đủ
print("Indexing enabled:", RUN_INDEXING)
print("Count by source:", dict(Counter(r["source_id"] for r in records)))
if not RUN_INDEXING:
    raise RuntimeError("DRY-RUN: chưa bật RUN_INDEXING=True; chưa tạo embeddings hoặc ChromaDB. Đây là trạng thái an toàn mặc định.")

Indexing enabled: True
Count by source: {'CDC-T2D-001': 4, 'CDC-RISK-002': 3, 'NIDDK-DIAG-003': 19, 'NIDDK-T2D-004': 11, 'WHO-DIAB-005': 5, 'LC-HBA1C-006': 11, 'LC-STAGES-007': 13}


## 6. Sinh Embeddings (lần đầu có thể tải model; CPU)
E5 sử dụng đầu vào với `passage:` và `normalize_embeddings=True`. Mỗi vector phải hữu hạn, có cùng dimension. **Không gửi văn bản ra API đám mây**.

In [9]:
doc_inputs=["passage: " + r["text"] for r in records]
embeddings=model.encode(doc_inputs,batch_size=16,show_progress_bar=True,convert_to_numpy=True,normalize_embeddings=True)
assert embeddings.shape==(len(records),model.get_sentence_embedding_dimension())
assert np.isfinite(embeddings).all()
assert np.allclose(np.linalg.norm(embeddings,axis=1),1,atol=1e-3)
print("Embeddings generated:",embeddings.shape)
print("Average vector norm:",float(np.linalg.norm(embeddings,axis=1).mean()))

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Embeddings generated: (66, 384)
Average vector norm: 1.0


C:\Users\ThinkPad\AppData\Local\Temp\ipykernel_22516\1475266770.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  assert embeddings.shape==(len(records),model.get_sentence_embedding_dimension())


## 7. Lưu ChromaDB ở máy người dùng
Dùng `PersistentClient` và cosine distance. Giữ snapshot hash; nếu đã có dữ liệu khác phiên bản, notebook sẽ dừng thay vì ghi đè. Metadata giữ nguồn, URL và loại văn bản gốc/ghi chú.

In [10]:
snapshot_items=sorted((r["chunk_id"],r["chunk_sha256"],r["source_id"],r["content_origin"]) for r in records)
snapshot_hash=hashlib.sha256(json.dumps(snapshot_items,ensure_ascii=False,separators=(",",":")).encode("utf-8")).hexdigest()
VECTOR_PATH.mkdir(parents=True,exist_ok=True)
client=chromadb.PersistentClient(path=str(VECTOR_PATH))
collection=client.get_or_create_collection(
    name=COLLECTION_NAME,
    configuration={"hnsw":{"space":"cosine"}},
    metadata={"model_id":MODEL_ID,"index_version":"v1","source_snapshot":snapshot_hash,"embedding_dim":int(embeddings.shape[1])}
)
meta=collection.metadata or {}
assert meta.get("model_id")==MODEL_ID, "Model của collection cũ khác — tạo collection mới"
assert meta.get("source_snapshot")==snapshot_hash, "Dữ liệu đã thay đổi — tạo collection tên mới thay vì overwrite"
assert int(meta.get("embedding_dim",0))==embeddings.shape[1], "Dimension không khớp"

metadatas=[]
for r in records:
    metadatas.append({
        "source_id":r["source_id"],
        "source_publisher":r.get("publisher", ""),
        "canonical_url":r["canonical_url"],
        "section":r["section"],
        "document_version":r["document_version"],
        "document_sha256":r["document_sha256"],
        "chunk_sha256":r["chunk_sha256"],
        "parent_chunk_id":r["parent_chunk_id"],
        "content_origin":r["content_origin"],
        "language":r.get("language", ""),
        "source_retrieved_at_utc":r.get("source_retrieved_at_utc", ""),
        "model_id": MODEL_ID
    })

if collection.count()==0:
    collection.add(
        ids=[r["chunk_id"] for r in records],
        embeddings=embeddings.tolist(),
        documents=[r["text"] for r in records],
        metadatas=metadatas
    )
else:
    assert collection.count()==len(records), "Số record khác dữ liệu input — không tự ghi đè"
    stored=collection.get(ids=[r["chunk_id"] for r in records],include=["metadatas"])
    assert len(stored["ids"])==len(records), "Thiếu IDs trong Chroma"
    old={i:m["chunk_sha256"] for i,m in zip(stored["ids"],stored["metadatas"])}
    assert all(old.get(r["chunk_id"])==r["chunk_sha256"] for r in records), "Record thay đổi — cần index phiên bản mới"
print("Chroma collection:",collection.name)
print("Stored records:",collection.count())
print("Location:",VECTOR_PATH)
print("Source snapshot:",snapshot_hash)

Chroma collection: diabetes_kb_step3_e5small_v1
Stored records: 66
Location: C:\Users\ThinkPad\Multi-Agent_Du_doan_benh_tieu_duong\Giai-doan-4-Bo-sung-Nha-thuoc-Long-Chau\Giai-doan-4-Buoc-2-Jupyter-Document-Preparation\data\vector_store\chroma_step3
Source snapshot: cc064ec7187192c999418051cb2125a0510412843f8d0908e3d6842e4a71c7cd


## 8. Truy vấn thử Semantic Search (không LLM)
Dùng đúng `query: `; in ra **ID, nguồn, section, URL, cosine distance**, không in toàn bộ bài viết. Giá trị distance càng nhỏ càng gần về vector, **không** là mức độ chính xác y khoa. Chỉ đọc mô hình từ máy bạn.

In [11]:
queries=[
    "Những yếu tố nào liên quan đến tiểu đường type 2?",
    "HbA1c dùng để đánh giá tình trạng đường huyết như thế nào?",
    "What are the common risk factors for type 2 diabetes?",
]
query_embeddings=model.encode(["query: "+q for q in queries],normalize_embeddings=True,convert_to_numpy=True)
res=collection.query(
    query_embeddings=query_embeddings.tolist(),
    n_results=min(3,collection.count()),
    include=["metadatas","distances","documents"]
)
search_summary=[]
for j,q in enumerate(queries):
    print("\nQUESTION:",q)
    hits=[]
    for rid,m,d in zip(res["ids"][j],res["metadatas"][j],res["distances"][j]):
        print(f" - {m['source_id']} | distance={d:.4f} | section={m['section']} | {m['canonical_url']}")
        hits.append({"chunk_id":rid,"source_id":m["source_id"],"section":m["section"],"url":m["canonical_url"],"cosine_distance":float(d),"content_origin":m["content_origin"]})
    search_summary.append({"query":q,"hits":hits})
print("\nĐiểm tìm kiếm không phải xác suất đúng; cần gán nhãn câu hỏi để đánh giá Retrieval Bước 4")


QUESTION: Những yếu tố nào liên quan đến tiểu đường type 2?
 - LC-STAGES-007 | distance=0.1050 | section=Cách chẩn đoán các giai đoạn tiểu đường | https://nhathuoclongchau.com.vn/bai-viet/cac-giai-doan-cua-benh-tieu-duong-20966.html
 - LC-STAGES-007 | distance=0.1075 | section=Giai đoạn 3: Bệnh tiểu đường tuýp 2 | https://nhathuoclongchau.com.vn/bai-viet/cac-giai-doan-cua-benh-tieu-duong-20966.html
 - LC-STAGES-007 | distance=0.1138 | section=Cách phòng ngừa bệnh tiểu đường tiến triển | https://nhathuoclongchau.com.vn/bai-viet/cac-giai-doan-cua-benh-tieu-duong-20966.html

QUESTION: HbA1c dùng để đánh giá tình trạng đường huyết như thế nào?
 - LC-HBA1C-006 | distance=0.0752 | section=HbA1c bao nhiêu là tiền tiểu đường? | https://nhathuoclongchau.com.vn/bai-viet/hb-a1c-bao-nhieu-la-tien-tieu-duong-nhan-biet-va-cai-thien.html
 - LC-STAGES-007 | distance=0.0901 | section=Cách chẩn đoán các giai đoạn tiểu đường | https://nhathuoclongchau.com.vn/bai-viet/cac-giai-doan-cua-benh-tieu-duong-20

## 9. Kiểm tra cuối cùng, lưu report JSON và gửi kết quả cho tôi
Không lưu toàn văn tài liệu trong report; chỉ lưu metadata và khoảng cách. Nếu JSON bị lỗi hãy gửi traceback, không sửa model hay dữ liệu Test.

In [12]:
stored=collection.get(include=["metadatas"])
source_counts=dict(Counter(m["source_id"] for m in stored["metadatas"]))
assert len(source_counts)==7, f"Số nguồn trong vector DB chỉ còn {len(source_counts)}"
assert collection.count()==len(records)
qa={
    "all_seven_sources_indexed":len(source_counts)==7,
    "count_matches_prepared_records":collection.count()==len(records),
    "embedding_dimension_matches":embeddings.shape[1]==model.get_sentence_embedding_dimension(),
    "no_finite_errors":bool(np.isfinite(embeddings).all()),
    "three_queries_have_results":all(len(q["hits"])>0 for q in search_summary),
    "review_claims_present":all(review_by_source[k]["index_approved"].strip().lower()=="yes" for k in source_counts),
}
assert all(qa.values()),f"QA failed: {qa}"
OUT_DIR.mkdir(parents=True,exist_ok=True)
report={
    "experiment":"EXP-RAG-EMBED-STEP3-001",
    "stage":"phase4_step3_embeddings_chromadb",
    "created_at_utc":datetime.now(timezone.utc).isoformat(),
    "status":"local_index_created_needs_retrieval_evaluation",
    "original_chunks":len(chunks),
    "records_indexed":collection.count(),
    "sources_indexed":source_counts,
    "embedding_model":MODEL_ID,
    "embedding_dimension":int(embeddings.shape[1]),
    "python_version":sys.version.split()[0],
    "chromadb_version":chromadb.__version__,
    "sentence_transformers_version":sentence_transformers.__version__,
    "collection":COLLECTION_NAME,
    "vector_store_path":str(VECTOR_PATH),
    "source_snapshot_sha256":snapshot_hash,
    "review_status":"self_declared_only_not_independently_verified",
    "qa":qa,
    "queries":search_summary,
    "llm_called":False,
    "model_retrained":False,
    "nhanes_test_reopened":False,
    "clinical_validation":False,
}
report_path=OUT_DIR/"step3_embedding_chroma_report.json"
report_path.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding="utf-8")
print("===== GIAI ĐOẠN 4 — BƯỚC 3: EMBEDDING & CHROMADB =====")
print("Original chunks:",len(chunks))
print("Indexed records:",collection.count())
print("Sources:",len(source_counts),"/7")
print("Embedding dimension:",embeddings.shape[1])
print("Collection:",COLLECTION_NAME)
print("QA:","PASS" if all(qa.values()) else "FAIL")
print("Report:",report_path)
print("LLM: NO | ML TRAINING: NO | NHANES TEST OPENED: NO")
print("===== KẾT THÚC — GỬI OUTPUT VÀ JSON CHO TÔI =====")

===== GIAI ĐOẠN 4 — BƯỚC 3: EMBEDDING & CHROMADB =====
Original chunks: 66
Indexed records: 66
Sources: 7 /7
Embedding dimension: 384
Collection: diabetes_kb_step3_e5small_v1
QA: PASS
Report: C:\Users\ThinkPad\Multi-Agent_Du_doan_benh_tieu_duong\Giai-doan-4-Bo-sung-Nha-thuoc-Long-Chau\Giai-doan-4-Buoc-2-Jupyter-Document-Preparation\data\outputs\embedding_chroma_step3\step3_embedding_chroma_report.json
LLM: NO | ML TRAINING: NO | NHANES TEST OPENED: NO
===== KẾT THÚC — GỬI OUTPUT VÀ JSON CHO TÔI =====


C:\Users\ThinkPad\AppData\Local\Temp\ipykernel_22516\1027826688.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  "embedding_dimension_matches":embeddings.shape[1]==model.get_sentence_embedding_dimension(),
